# UAV Pathfinding + LoRA fine-tuning of a ~1B LLM -- Colab quickstart

Fine-tunes a small open LLM (default: `meta-llama/Llama-3.2-1B-Instruct`) with LoRA to solve a simple UAV grid pathfinding task (fly from `S` to `G` avoiding obstacles `#`), and scores it against the BFS-optimal path.

**Before running:** Runtime -> Change runtime type -> GPU (T4 is fine, that's what the default config is tuned for).

If you're not opening this notebook from inside an already-cloned repo, edit `REPO_URL` in the next cell first.

In [ ]:
REPO_URL = "https://github.com/vuhoangviet0808/llm-uav-.git"  # <-- edit this
REPO_DIR = REPO_URL.rstrip('/').split('/')[-1].replace('.git', '')

import os
if not os.path.isdir(REPO_DIR):
    !git clone $REPO_URL
%cd $REPO_DIR

Cloning into 'llm-uav-'...
remote: Enumerating objects: 30, done.
remote: Counting objects: 100% (30/30), done.
remote: Compressing objects: 100% (24/24), done.
remote: Total 30 (delta 4), reused 30 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (30/30), 21.93 KiB | 21.93 MiB/s, done.
Resolving deltas: 100% (4/4), done.
/content/llm-uav-/llm-uav-/llm-uav-


In [ ]:
!nvidia-smi

Sun Aug 16 05:28:38 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!pip install -q -r requirements.txt

## (Only if using a gated model like Llama-3.2-1B-Instruct)

1. Accept the license at https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct ("Agree and access repository").
2. Create a token at https://huggingface.co/settings/tokens (read access is enough).
3. Run the cell below and paste the token when prompted.

Skip this if you switched `configs/default.yaml` -> `model.name` to an ungated model such as `Qwen/Qwen2.5-0.5B-Instruct` or `HuggingFaceTB/SmolLM2-1.7B-Instruct`.

In [ ]:
from huggingface_hub import login
login()

## 1. Generate the dataset (BFS ground truth)

In [ ]:
!python -m src.data_gen --config configs/default.yaml

[train] wrote 3000 examples to data/train.jsonl (avg optimal path length = 7.3)
[val] wrote 200 examples to data/val.jsonl (avg optimal path length = 7.2)
[test] wrote 200 examples to data/test.jsonl (avg optimal path length = 7.0)


## 2. Evaluate the base model zero-shot (before fine-tuning) -- baseline for comparison

`--limit 50` keeps this quick; drop it to evaluate the full test set.

In [ ]:
!python -m src.evaluate --config configs/default.yaml \
    --report_path outputs/eval_report_base.json --limit 50

config.json: 100% 660/660 [00:00<00:00, 2.50MB/s]
tokenizer_config.json: 100% 7.30k/7.30k [00:00<00:00, 4.35MB/s]
vocab.json: 100% 2.78M/2.78M [00:00<00:00, 15.1MB/s]
merges.txt: 100% 1.67M/1.67M [00:00<00:00, 13.8MB/s]
tokenizer.json: 100% 7.03M/7.03M [00:00<00:00, 21.5MB/s]

model.safetensors: downloading bytes:   0% 0.00/3.09G [00:00<?, ?B/s]
model.safetensors: downloading bytes:   2% 71.4M/3.09G [00:02<00:47, 63.2MB/s, 4.81MB/s  ]
model.safetensors: downloading bytes:   3% 91.8M/3.09G [00:02<00:35, 85.5MB/s, 6.68MB/s  ]
model.safetensors: downloading bytes:   6% 171M/3.09G [00:02<00:17, 165MB/s, 13.5MB/s  ]
model.safetensors: downloading bytes:   6% 200M/3.09G [00:03<00:18, 158MB/s, 17.1MB/s  ]
model.safetensors: downloading bytes:   8% 236M/3.09G [00:03<00:17, 164MB/s, 18.9MB/s  ]
model.safetensors: downloading bytes:   8% 258M/3.09G [00:03<00:16, 173MB/s, 21.1MB/s  ]
model.safetensors: downloading bytes:  10% 305M/3.09G [00:03<00:14, 194MB/s, 24.3MB/s  ]
model.safetensors: recons

## 3. LoRA fine-tune

Edit `configs/default.yaml` first if you want to change the model, grid sizes, dataset size, batch size, etc. Defaults are sized for a free T4 (16GB) Colab GPU.

In [ ]:
!python -m src.train --config configs/default.yaml

Loading base model 'Qwen/Qwen2.5-1.5B-Instruct' ...
Loading weights: 100% 338/338 [00:00<00:00, 5022.51it/s]
Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/content/llm-uav-/llm-uav-/llm-uav-/src/train.py", line 88, in <module>
    main()
  File "/content/llm-uav-/llm-uav-/llm-uav-/src/train.py", line 42, in main
    model = apply_lora(
            ^^^^^^^^^^^
  File "/content/llm-uav-/llm-uav-/llm-uav-/src/model.py", line 60, in apply_lora
    model = get_peft_model(model, config)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py", line 122, in get_peft_model
    return MODEL_TYPE_TO_PEFT_MODEL_MAPPING[peft_config.task_type](
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/peft/peft_model.py", line 1955, in __init__
    super().__init__(model, peft_config, ad

## 4. Evaluate the fine-tuned model

In [ ]:
!python -m src.evaluate --config configs/default.yaml \
    --adapter_dir outputs/lora-pathfinding \
    --report_path outputs/eval_report_finetuned.json

Loading weights: 100% 338/338 [00:00<00:00, 5586.79it/s]
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_http.py", line 795, in hf_raise_for_status
    response.raise_for_status()
  File "/usr/local/lib/python3.12/dist-packages/httpx/_models.py", line 829, in raise_for_status
    raise HTTPStatusError(message, request=request, response=self)
httpx.HTTPStatusError: Client error '404 Not Found' for url 'https://huggingface.co/outputs/lora-pathfinding/resolve/main/adapter_config.json'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/404

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/peft/config.py", line 317, in _get_peft_type
    config_file = hf_hub_download(
                  ^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py", line 88, i

In [ ]:
import json
base = json.load(open("outputs/eval_report_base.json"))["summary"]
ft = json.load(open("outputs/eval_report_finetuned.json"))["summary"]
print("BASE      :", base)
print("FINE-TUNED:", ft)

FileNotFoundError: [Errno 2] No such file or directory: 'outputs/eval_report_finetuned.json'

## 5. Visualize one example (BFS-optimal path vs. the model's actual path)

In [ ]:
!python -m src.visualize --config configs/default.yaml --split test --index 0 \
    --eval_report outputs/eval_report_finetuned.json --out outputs/example.png

from IPython.display import Image
Image("outputs/example.png")

## Notes

- Metrics reported (see `README.md` for details): `success_rate`, `avg_length_ratio_on_success` (1.0 = as short as the BFS-optimal path), `invalid_move_rate`, `unparseable_rate`.
- If you hit an out-of-memory error on T4, lower `train.per_device_train_batch_size` in `configs/default.yaml` and raise `train.gradient_accumulation_steps` to compensate, or lower `data.grid_size_range` / `model.max_length`.
- Before touching the real model, you can sanity-check the whole pipeline offline in seconds/minutes with `configs/smoke_test.yaml` (see `scripts/smoke_test.sh`) -- it uses a tiny randomly-initialized model and a char-level tokenizer, no GPU or internet needed.